# Feature Engineering — demanda por tramo horario

Fase 5 de la guía. Partimos de las conclusiones de `eda.ipynb` y aplicamos siempre la
misma transformación a train y a test.

**La lógica de cada variable vive en `src/feature_engineering.py`**, no aquí — este
notebook solo llama a esas funciones y verifica el resultado. Es la misma función que se
usará para predecir una fecha real en el futuro (`construir_features(fecha, tramo)`), así
que entrenamiento y producción calculan las variables exactamente igual.

Requiere el paquete `holidays` (`pip install -r requirements.txt`).

Variables:
1. `trimestre` y `dias_desde_inicio` (tendencia).
2. `grupo_dia` y `temporada` (saltos semanales y estación).
3. `tramo_tarde` (versión 0/1 de `tramo`).
4. `es_festivo` (calendario oficial de Andalucía).
5. `es_vispera_festivo` y `es_fecha_comercial` (vísperas y fechas señaladas tipo San
   Valentín).
6. `es_cierre` — **marcador, no feature**: días en que el spa no abre (se explica en su
   sección).

Se evaluaron también *lags* (citas de fechas anteriores) y por ahora quedan fuera — el
porqué está en la sección 7.

## 0. Qué va aquí y qué venía ya de `transform.ipynb`

Las columnas de calendario del dataset (`dia_semana`, `mes`, `es_finde`...) son la
descomposición directa de la fecha: se crearon en `transform.ipynb`, antes del split, para
que la EDA pudiera agrupar por ellas desde el principio. Las de este notebook nacen de lo
que la EDA encontró (o de fuentes externas, como los festivos) — por eso llegan después.

Es normal que alguna variable nueva deje obsoleta a una de calendario — pasa con
`es_finde` → `grupo_dia` (sección 2). Y como estas variables no existían durante la EDA,
la sección 8 comprueba su relación con el target antes de guardarlas.

In [10]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path('..').resolve()))
from src.utils.feature_engineering import (
    anadir_tendencia,
    anadir_variables_negocio,
    anadir_tramo_tarde,
    anadir_festivos,
    anadir_vispera_y_comercial,
    anadir_cierre,
    primer_domingo_mayo,
    FESTIVOS_ANDALUCIA,
)

pd.set_option('display.max_columns', None)

TRAIN_PATH = Path('../data/processed/train.csv')
TEST_PATH = Path('../data/processed/test.csv')
TRAIN_OUT = Path('../data/processed/train_features.csv')
TEST_OUT = Path('../data/processed/test_features.csv')
TARGET = 'n_citas'

train = pd.read_csv(TRAIN_PATH, parse_dates=['fecha_cita'])
test = pd.read_csv(TEST_PATH, parse_dates=['fecha_cita'])

print(f"Train: {train.shape} | {train['fecha_cita'].min().date()} -> {train['fecha_cita'].max().date()}")
print(f"Test:  {test.shape} | {test['fecha_cita'].min().date()} -> {test['fecha_cita'].max().date()}")

Train: (1252, 9) | 2024-05-09 -> 2026-01-24
Test:  (314, 9) | 2026-01-25 -> 2026-06-30


## 1. Trimestre y tendencia

`trimestre` sale directo de la fecha.

`dias_desde_inicio` es la variable de tendencia que pedía la EDA (§3.5): las variables de
calendario se repiten cada año, así que para ellas mayo de 2024 y mayo de 2025 son
idénticos — cuando en realidad hubo 52 citas frente a 293. Este contador (días desde la
primera fecha de train) le da al modelo el eje de "cuánto ha crecido el negocio". La fecha
de referencia se fija con train y se reutiliza en test.

*(Se descartó la codificación seno/coseno de mes y día que hubo en una versión anterior:
complejidad extra que los modelos de árboles no necesitan.)*

*Aviso para Modelado: los árboles no extrapolan esta variable más allá del rango visto en
train — al predecir lejos se quedan en el nivel del final del histórico. Un modelo lineal
sí proyecta la tendencia.*

In [11]:
FECHA_REFERENCIA = train['fecha_cita'].min()  # "aprendida" en train, reutilizada en test

train = anadir_tendencia(train, FECHA_REFERENCIA)
test = anadir_tendencia(test, FECHA_REFERENCIA)

train[['fecha_cita', 'trimestre', 'dias_desde_inicio']].head()

,fecha_cita,trimestre,dias_desde_inicio
0,2024-05-09,2,0
1,2024-05-09,2,0
2,2024-05-10,2,1
3,2024-05-10,2,1
4,2024-05-11,2,2


## 2. `grupo_dia` y `temporada`

Las dos salen directas de la EDA. La demanda semanal va a saltos, no en gradiente (plana
de lunes a jueves, sube el viernes por la tarde, pico el fin de semana, §3.2) →
`grupo_dia` con tres categorías. Y la estación pesa más que el número del mes — diciembre
y enero se comportan igual pero son 12 y 1 en la escala (§3.3) → `temporada`.

`grupo_dia` deja obsoleta a `es_finde`: es la misma información pero sin perder al
viernes. Se mantienen ambas de momento y `es_finde` se elimina en Preprocesado. Con `mes`
y `temporada` no pasa lo mismo: `mes` tiene el detalle fino y `temporada` es su agrupación
de negocio — se quedan las dos.

In [12]:
train = anadir_variables_negocio(train)
test = anadir_variables_negocio(test)

train.groupby('grupo_dia')[TARGET].mean().round(2)

grupo_dia
entre_semana     2.50
fin_de_semana    5.26
viernes          4.20
Name: n_citas, dtype: float64

## 3. `tramo_tarde`

La variable con más peso según la EDA (§3.1: la tarde casi dobla a la mañana), en versión
0/1 para los modelos que no aceptan texto. Contiene exactamente la misma información que
`tramo` — aquí se conservan las dos por legibilidad, y `tramo` se elimina en Preprocesado.

In [13]:
train = anadir_tramo_tarde(train)
test = anadir_tramo_tarde(test)

train[['tramo', 'tramo_tarde']].drop_duplicates()

,tramo,tramo_tarde
0,mañana,0
1,tarde,1


## 4. `es_festivo`

Usamos la librería `holidays` con el calendario oficial de Andalucía (`subdiv='AN'`):
trae los festivos nacionales, el Día de Andalucía, Jueves y Viernes Santo en su fecha
correcta cada año, y los traslados a lunes cuando un festivo cae en domingo. En
`src/feature_engineering.py` se expande automáticamente a cualquier año que se consulte —
no hace falta fijar un rango de años como aquí, eso es importante para cuando esta misma
función se use para predecir una fecha real futura.

Lo que ninguna librería trae son las 2 fiestas locales de Sevilla capital (las decide el
Ayuntamiento cada año) — `FESTIVOS_LOCALES_SEVILLA` (en `src/feature_engineering.py`)
queda vacía, lista para rellenar consultando el BOJA.

Ojo con lo que cabe esperar de esta variable: la EDA (§5.3) encontró que el spa **cierra**
en algunos festivos (Navidad, Año Nuevo, Reyes), así que "festivo" mezcla días de cierre
con días de posible demanda extra — no va a ser una señal limpia.

In [14]:
import holidays

anios = range(train['fecha_cita'].dt.year.min(), test['fecha_cita'].dt.year.max() + 1)

train = anadir_festivos(train)
test = anadir_festivos(test)

# Objeto aparte, solo para listar y verificar los festivos calculados en el rango de datos
# (la columna es_festivo en sí se calcula con FESTIVOS_ANDALUCIA, que se expande a cualquier año)
festivos_verificacion = holidays.Spain(subdiv='AN', years=list(anios))
print(f"Festivos calculados para {list(anios)}: {len(festivos_verificacion)}")
for fecha, nombre in sorted(festivos_verificacion.items()):
    print(fecha, '-', nombre)

Festivos calculados para [2024, 2025, 2026]: 36
2024-01-01 - Año Nuevo
2024-01-06 - Epifanía del Señor
2024-02-28 - Día de Andalucía
2024-03-28 - Jueves Santo
2024-03-29 - Viernes Santo
2024-05-01 - Fiesta del Trabajo
2024-08-15 - Asunción de la Virgen
2024-10-12 - Fiesta Nacional de España
2024-11-01 - Todos los Santos
2024-12-06 - Día de la Constitución Española
2024-12-09 - Lunes siguiente a Inmaculada Concepción
2024-12-25 - Natividad del Señor
2025-01-01 - Año Nuevo
2025-01-06 - Epifanía del Señor
2025-02-28 - Día de Andalucía
2025-04-17 - Jueves Santo
2025-04-18 - Viernes Santo
2025-05-01 - Fiesta del Trabajo
2025-08-15 - Asunción de la Virgen
2025-10-13 - Lunes siguiente a Fiesta Nacional de España
2025-11-01 - Todos los Santos
2025-12-06 - Día de la Constitución Española
2025-12-08 - Inmaculada Concepción
2025-12-25 - Natividad del Señor
2026-01-01 - Año Nuevo
2026-01-06 - Epifanía del Señor
2026-02-28 - Día de Andalucía
2026-04-02 - Jueves Santo
2026-04-03 - Viernes Santo
2026

## 5. `es_vispera_festivo` y `es_fecha_comercial`

Dos flags más, calculables para cualquier fecha futura solo con el calendario:

- `es_vispera_festivo`: día anterior a festivo. Plausible que la gente reserve spa cuando
  el día siguiente es libre. Sale del mismo calendario de la sección 4.
- `es_fecha_comercial`: fechas señaladas de regalo/pareja — el pico más alto de todo el
  histórico es el 14 de febrero (San Valentín), y en los datos originales existe incluso
  un producto "Ritual Día de la Madre". Incluimos San Valentín (fijo) y el Día de la Madre
  (primer domingo de mayo en España, se calcula solo).

Las dos se comprueban contra el target en la sección 8 antes de darlas por buenas.

In [15]:
train = anadir_vispera_y_comercial(train)
test = anadir_vispera_y_comercial(test)

# Solo para el print de verificación (las fechas comerciales reales se calculan dentro de la función)
fechas_comerciales_verificacion = set()
for anio in anios:
    fechas_comerciales_verificacion.add(pd.Timestamp(year=anio, month=2, day=14))
    fechas_comerciales_verificacion.add(primer_domingo_mayo(anio))

print("Fechas comerciales:", sorted(d.date() for d in fechas_comerciales_verificacion))
print(f"Vísperas de festivo en train: {train['es_vispera_festivo'].sum() // 2} días")

Fechas comerciales: [datetime.date(2024, 2, 14), datetime.date(2024, 5, 5), datetime.date(2025, 2, 14), datetime.date(2025, 5, 4), datetime.date(2026, 2, 14), datetime.date(2026, 5, 3)]
Vísperas de festivo en train: 20 días


### 5.1 ¿Hay más fechas comerciales que aporten señal?

Antes de dar `es_fecha_comercial` por cerrada, probamos otras fechas candidatas con
significado de regalo/consumo en España (Día del Padre, Día de la Mujer, Black Friday,
Cyber Monday, Nochevieja, Noche de San Juan) para ver si alguna muestra el mismo tipo de
repunte que San Valentín o el Día de la Madre — y si añadirlas reforzaría la señal o la
diluiría.

In [16]:
def ultimo_viernes_noviembre(anio):
    fin_nov = pd.Timestamp(year=anio, month=11, day=30)
    return fin_nov - pd.Timedelta(days=(fin_nov.weekday() - 4) % 7)


candidatas = {
    'San Valentín (14 feb) — ya incluida': [pd.Timestamp(a, 2, 14) for a in anios],
    'Día de la Madre (1er dom mayo) — ya incluida': [primer_domingo_mayo(a) for a in anios],
    'Día del Padre (19 marzo)': [pd.Timestamp(a, 3, 19) for a in anios],
    'Día de la Mujer (8 marzo)': [pd.Timestamp(a, 3, 8) for a in anios],
    'Black Friday (últ. viernes nov)': [ultimo_viernes_noviembre(a) for a in anios],
    'Cyber Monday (lunes tras BF)': [ultimo_viernes_noviembre(a) + pd.Timedelta(days=3) for a in anios],
    'Nochevieja (31 dic)': [pd.Timestamp(a, 12, 31) for a in anios],
    'Noche de San Juan (23 jun)': [pd.Timestamp(a, 6, 23) for a in anios],
}

fmin, fmax = train['fecha_cita'].min(), train['fecha_cita'].max()
base = train.groupby('tramo')[TARGET].mean().round(2)
print(f"Media global de referencia -> mañana: {base['mañana']} | tarde: {base['tarde']}\n")

for nombre, fechas in candidatas.items():
    en_train = sorted(f for f in fechas if fmin <= f <= fmax)
    if not en_train:
        print(f"{nombre:45s} sin fechas dentro del rango de train")
        continue
    medias = train[train['fecha_cita'].isin(en_train)].groupby('tramo')[TARGET].mean().round(2)
    print(f"{nombre:45s} n={len(en_train)}  mañana={medias.get('mañana', float('nan')):>5}  "
          f"tarde={medias.get('tarde', float('nan')):>5}")

Media global de referencia -> mañana: 2.52 | tarde: 4.54

San Valentín (14 feb) — ya incluida           n=1  mañana=  7.0  tarde= 15.0
Día de la Madre (1er dom mayo) — ya incluida  n=1  mañana=  4.0  tarde=  6.0
Día del Padre (19 marzo)                      n=1  mañana=  4.0  tarde=  6.0
Día de la Mujer (8 marzo)                     n=1  mañana= 11.0  tarde=  5.0
Black Friday (últ. viernes nov)               n=2  mañana=  1.0  tarde=  6.5
Cyber Monday (lunes tras BF)                  n=2  mañana=  0.5  tarde=  3.0
Nochevieja (31 dic)                           n=2  mañana=  2.0  tarde=  3.5
Noche de San Juan (23 jun)                    n=2  mañana=  1.0  tarde=  2.5


**Lectura — ninguna candidata nueva se incorpora, y además destapa un problema de fondo:**

- **La mayoría no muestra repunte.** Black Friday, Cyber Monday, Nochevieja y Noche de San
  Juan quedan igual o por debajo de la media global; Día del Padre se queda justo en la
  media.
- **Día de la Mujer es la excepción llamativa, y por eso mismo sospechosa**: mañana=11
  frente a una media de 2,52 (×4), pero tarde=5, prácticamente en la media (4,54). Un
  repunte real de "fecha de regalo" debería notarse en ambos tramos, como pasa con San
  Valentín — que suba solo uno de los dos con un único dato detrás huele más a casualidad
  de un día concreto que a patrón. No se incorpora.
- **El hallazgo de fondo es otro: `es_fecha_comercial` tiene un tamaño de muestra
  minúsculo.** Train solo cubre un San Valentín (14/02/2025) y un Día de la Madre
  (04/05/2025) — los demás años caen fuera del rango de train. La "señal fuerte" de la
  sección 8 está sostenida por **2 días sueltos (4 filas)**, y el 14/02/2025 es
  literalmente el outlier de tarde más alto de todo el dataset (15 citas, EDA §4.2). No es
  que la variable esté mal planteada — Valentín y el Día de la Madre son fechas de regalo
  reales, una con producto propio en el catálogo — pero con un solo caso por fecha no se
  distingue "patrón real" de "casualidad de un día muy bueno". Se mantiene, con esta
  reserva explícita, hasta que el histórico crezca y haya más de un año por fecha para
  confirmarlo.

### 5.2 Validación con el histórico completo (de cara al modelo final)

La reserva de la §5.1 era de tamaño de muestra: solo 1 ocurrencia de cada fecha dentro de
train. Pero el producto final no se queda con este split — cuando se entrene el modelo
definitivo, **todo el histórico actual (train + test) pasará a ser el nuevo train**, y lo
que hoy es test se sustituirá por fechas realmente futuras a predecir. Así que tiene
sentido comprobar cómo se ven estos patrones con **train + test juntos**, simulando esa
situación futura.

**Importante:** esto es solo una validación de cara a esa retrain futura, no cambia nada
del pipeline actual — `train_features.csv`/`test_features.csv` se generan igual que
siempre, sin tocar test. Es una comprobación aparte, con su propia carga de datos.

In [17]:
# Carga aparte, solo para esta validación — no sustituye a train/test del pipeline
_completo_validacion = pd.concat(
    [pd.read_csv(TRAIN_PATH, parse_dates=['fecha_cita']), pd.read_csv(TEST_PATH, parse_dates=['fecha_cita'])],
    ignore_index=True,
)
_fmin, _fmax = _completo_validacion['fecha_cita'].min(), _completo_validacion['fecha_cita'].max()
_base = _completo_validacion.groupby('tramo')[TARGET].mean().round(2)

print(f"Histórico completo: {_fmin.date()} -> {_fmax.date()} ({len(_completo_validacion)} filas)")
print(f"Media global -> mañana: {_base['mañana']} | tarde: {_base['tarde']}\n")

for nombre, fechas in candidatas.items():
    en_rango = sorted(f for f in fechas if _fmin <= f <= _fmax)
    if not en_rango:
        continue
    sub = _completo_validacion[_completo_validacion['fecha_cita'].isin(en_rango)]
    medias = sub.groupby('tramo')[TARGET].mean().round(2)
    fechas_str = ', '.join(f.strftime('%Y-%m-%d') for f in en_rango)
    print(f"{nombre:35s} n={len(en_rango)}  mañana={medias.get('mañana', float('nan')):>5}  "
          f"tarde={medias.get('tarde', float('nan')):>5}   [{fechas_str}]")

Histórico completo: 2024-05-09 -> 2026-06-30 (1566 filas)
Media global -> mañana: 2.81 | tarde: 4.91

San Valentín (14 feb) — ya incluida n=2  mañana=  8.5  tarde= 17.5   [2025-02-14, 2026-02-14]
Día de la Madre (1er dom mayo) — ya incluida n=2  mañana=  6.5  tarde=  8.0   [2025-05-04, 2026-05-03]
Día del Padre (19 marzo)            n=2  mañana=  4.5  tarde=  5.5   [2025-03-19, 2026-03-19]
Día de la Mujer (8 marzo)           n=2  mañana=  8.5  tarde=  7.0   [2025-03-08, 2026-03-08]
Black Friday (últ. viernes nov)     n=2  mañana=  1.0  tarde=  6.5   [2024-11-29, 2025-11-28]
Cyber Monday (lunes tras BF)        n=2  mañana=  0.5  tarde=  3.0   [2024-12-02, 2025-12-01]
Nochevieja (31 dic)                 n=2  mañana=  2.0  tarde=  3.5   [2024-12-31, 2025-12-31]
Noche de San Juan (23 jun)          n=3  mañana= 0.67  tarde=  4.0   [2024-06-23, 2025-06-23, 2026-06-23]


**Lectura — con un año más de datos, el panorama cambia para las dos fechas buenas:**

- **San Valentín queda confirmado, no era casualidad.** Ahora hay 2 años: 2025 (7/15) y
  **2026 (10/20) — todavía más alto**. Las dos veces muy por encima de la media global
  (2,81/4,91), en ambos tramos, y subiendo. Es el candidato más sólido de todos.
- **Día de la Madre también se confirma y crece**: 2025 (4/6) y 2026 (9/10), las dos veces
  por encima de la media y con la misma dirección. Con dos años consistentes, deja de ser
  "un dato suelto".
- **El resto sigue sin mostrar patrón, ahora con más evidencia todavía de que es ruido**:
  Día del Padre se mantiene pegado a la media los dos años; **Día de la Mujer se
  contradice entre años** (2025 dispara mañana pero no tarde, 2026 al revés — justo el
  comportamiento errático que hacía sospechar en §5.1); Black Friday, Cyber Monday,
  Nochevieja y Noche de San Juan (con 3 años ya) siguen sin despegar de la media o por
  debajo.

**Conclusión para el modelo final**: cuando se reentrene con todo el histórico, San
Valentín y Día de la Madre pasan de "señal con reserva" a **señal confirmada con 2 años
consecutivos** — no haría falta ningún cambio en `es_fecha_comercial`, las fechas ya
elegidas eran las correctas. El resto de candidatas se descartan con más confianza
todavía que en train solo.

## 6. `es_cierre` — marcador, no feature del modelo

La EDA (§5.3) encontró que el spa no abre en Navidad, Año Nuevo y Reyes (cero citas ambos
años). Marcamos esos días, pero **esta columna no debe entrar al modelo como feature**: si
se sabe que está cerrado, no tiene sentido pedirle al modelo que estime las citas — la
respuesta es 0 por regla de negocio, no por predicción.

Su utilidad es otra:

- **En Preprocesado**: excluir estas filas de train. Si se quedan, el modelo aprende "los
  25 de diciembre la demanda es bajísima" cuando en realidad es que no se abrió, y eso
  contamina lo que aprende de los días normales de invierno.
- **En producción**: regla previa al modelo — si la fecha está marcada como cierre, se
  devuelve 0 y no se llama al modelo.

Solo marcamos las 3 fechas **confirmadas por el EDA** (se repiten los dos años),
`CIERRES_RECURRENTES` en `src/feature_engineering.py`. Los bloques de marzo y noviembre de
2025 que vimos también a cero NO se marcan porque no sabemos si fueron cierres o
casualidad — `CIERRES_CONOCIDOS` (mismo fichero) queda como lista manual para añadirlos si
el negocio lo confirma, igual que hicimos con los festivos locales.

In [18]:
train = anadir_cierre(train)
test = anadir_cierre(test)

print(f"Días marcados como cierre en train: {train['es_cierre'].sum() // 2}")
print(f"Días marcados como cierre en test:  {test['es_cierre'].sum() // 2}")

Días marcados como cierre en train: 6
Días marcados como cierre en test:  0


## 7. Por qué no incluimos *lags* (de momento)

Un lag ("cuántas citas hubo hace X días en esta franja") predice bien — probamos 7, 14 y
364 días y la correlación con el target rondaba 0,55 — pero lo dejamos fuera por ahora:

- **Solo se puede calcular si la fecha a la que apunta ya ha pasado.** `lag_7` sirve para
  predecir hasta 7 días vista; este proyecto necesita también horizontes largos.
- **Un lag largo valdría para horizontes largos, pero se come el train**: con `lag_364` el
  58% de las filas se quedaba sin valor (no hay "hace un año" para el primer año del
  histórico).
- **En producción exige tener el histórico de reservas consultable automáticamente** cada
  vez que se predice — algo que aún no está decidido.

No es un descarte definitivo: cuando se concrete el horizonte de cada uso del modelo y la
disponibilidad del dato, se retoma. Mientras, la tendencia y la estacionalidad quedan
cubiertas por `dias_desde_inicio`, `mes`, `temporada` y `grupo_dia`, que se calculan para
cualquier fecha futura sin depender de nada.

## 8. Verificación contra el target

Estas variables se crearon después de la EDA, así que su relación con `n_citas` no se ha
comprobado aún. Chequeo rápido antes de guardarlas — si alguna no mostrara relación
ninguna, habría que revisarla en vez de pasarla al modelo sin más. Para `es_cierre` la
comprobación es distinta: debe salir con media ≈ 0, confirmando que marca bien los días
cerrados.

In [19]:
print("Media de n_citas: festivo vs. no festivo (por tramo)")
print(train.groupby(['tramo', 'es_festivo'])[TARGET].mean().round(2))

print("\nMedia de n_citas por temporada (por tramo)")
print(train.groupby(['tramo', 'temporada'])[TARGET].mean().round(2))

print("\nMedia de n_citas: víspera de festivo vs. resto (por tramo)")
print(train.groupby(['tramo', 'es_vispera_festivo'])[TARGET].mean().round(2))

print("\nMedia de n_citas: fecha comercial vs. resto (por tramo)")
print(train.groupby(['tramo', 'es_fecha_comercial'])[TARGET].mean().round(2))

print("\nMedia de n_citas en días marcados como cierre (debe ser ~0)")
print(train.groupby('es_cierre')[TARGET].mean().round(2))

print("\nCorrelación con n_citas de las variables numéricas nuevas")
print(train[['dias_desde_inicio', TARGET]].corr()[TARGET].round(3))

Media de n_citas: festivo vs. no festivo (por tramo)
tramo   es_festivo
mañana  False         2.52
        True          2.70
tarde   False         4.54
        True          4.65
Name: n_citas, dtype: float64

Media de n_citas por temporada (por tramo)
tramo   temporada
mañana  invierno     2.97
        otoño        2.78
        primavera    2.78
        verano       1.76
tarde   invierno     5.23
        otoño        4.80
        primavera    4.57
        verano       3.73
Name: n_citas, dtype: float64

Media de n_citas: víspera de festivo vs. resto (por tramo)
tramo   es_vispera_festivo
mañana  False                 2.54
        True                  2.05
tarde   False                 4.54
        True                  4.65
Name: n_citas, dtype: float64

Media de n_citas: fecha comercial vs. resto (por tramo)
tramo   es_fecha_comercial
mañana  False                  2.51
        True                   5.50
tarde   False                  4.52
        True                  10.50
Name:

**Lectura:**

- `es_fecha_comercial`: la señal nueva más fuerte con diferencia — esas fechas duplican
  con creces la media (mañana 5,5 vs. 2,5; tarde 10,5 vs. 4,5). Se queda, **pero con
  reserva**: son solo 2 fechas con 1 ocurrencia cada una en train (ver §5.1) — la señal es
  real en el sentido de que esos días pasaron, pero el tamaño de muestra es demasiado
  pequeño para hablar de patrón confirmado todavía.
- `temporada`: patrón claro en ambos tramos (invierno > otoño/primavera > verano). Útil.
- `dias_desde_inicio` (0,28): señal moderada, confirma la tendencia.
- `es_cierre`: media exactamente 0,00 en los días marcados — el marcador funciona.
- `es_festivo`: casi no mueve la media — como anticipaba la EDA (§5.3), los festivos de
  cierre y los de demanda extra se cancelan. Se mantiene pero es candidata a descartar.
- `es_vispera_festivo`: no muestra señal (la mañana incluso baja un poco). La hipótesis
  "reservan la víspera" no se cumple en estos datos. Se guarda igualmente, pero junto a
  `es_festivo` es la otra candidata clara a eliminar en Preprocesado si se simplifica.

## 9. Guardado

Notas para Preprocesado, todas juntas para no perder ninguna:

**Columnas redundantes a eliminar:**

| Eliminar | En favor de | Motivo |
|---|---|---|
| `tramo` | `tramo_tarde` | misma información, texto vs. 0/1 |
| `nombre_dia` | `dia_semana` | misma información, texto vs. número |
| `es_finde` | `grupo_dia` | contenida en `grupo_dia`, no distingue el viernes |

(`mes`/`temporada` y `dia_semana`/`grupo_dia` no son duplicados: detalle fino +
agrupación de negocio, se quedan ambos pares.)

**Tratamiento especial:**

- `es_cierre`: **no pasar al modelo como feature**. Excluir sus filas de train, y en
  producción devolver 0 por regla sin llamar al modelo (sección 6).
- `es_festivo` y `es_vispera_festivo`: señal débil verificada — primeras candidatas a
  eliminar si hay que simplificar.

In [21]:
TRAIN_OUT.parent.mkdir(parents=True, exist_ok=True)
train.to_csv(TRAIN_OUT, index=False)
test.to_csv(TEST_OUT, index=False)

print(f"Guardado train con features en: {TRAIN_OUT.resolve()}")
print(f"Guardado test con features en:  {TEST_OUT.resolve()}")
print(f"\nColumnas finales ({train.shape[1]}): {list(train.columns)}")
train.tail()

Guardado train con features en: D:\Usuarios\Emilio\Documentos\GitHub\ML_Spa\ML_Spa_M003\data\processed\train_features.csv
Guardado test con features en:  D:\Usuarios\Emilio\Documentos\GitHub\ML_Spa\ML_Spa_M003\data\processed\test_features.csv

Columnas finales (18): ['fecha_cita', 'tramo', 'n_citas', 'dia_semana', 'nombre_dia', 'es_finde', 'mes', 'anio', 'semana_iso', 'trimestre', 'dias_desde_inicio', 'grupo_dia', 'temporada', 'tramo_tarde', 'es_festivo', 'es_vispera_festivo', 'es_fecha_comercial', 'es_cierre']


,fecha_cita,tramo,n_citas,dia_semana,nombre_dia,es_finde,mes,anio,semana_iso,trimestre,dias_desde_inicio,grupo_dia,temporada,tramo_tarde,es_festivo,es_vispera_festivo,es_fecha_comercial,es_cierre
1247,2026-01-22,tarde,3,3,Thursday,False,1,2026,4,1,623,entre_semana,invierno,1,False,False,False,False
1248,2026-01-23,mañana,3,4,Friday,False,1,2026,4,1,624,viernes,invierno,0,False,False,False,False
1249,2026-01-23,tarde,7,4,Friday,False,1,2026,4,1,624,viernes,invierno,1,False,False,False,False
1250,2026-01-24,mañana,8,5,Saturday,True,1,2026,4,1,625,fin_de_semana,invierno,0,False,False,False,False
1251,2026-01-24,tarde,12,5,Saturday,True,1,2026,4,1,625,fin_de_semana,invierno,1,False,False,False,False


## 10. Cómo se usará esto para predecir una fecha real

Todo lo de este notebook son llamadas a funciones de `src/feature_engineering.py` sobre
todo el histórico a la vez. Para predecir una fecha nueva en producción, la misma lógica
se usa con `construir_features(fecha, tramo)` — una fecha, un tramo, una fila. **`tramo`
tiene que darse como input, no se puede derivar de la fecha** (no hay forma de saber si se
pregunta por la mañana o la tarde solo con el día); para predecir un día completo se llama
dos veces, una por tramo. Ejemplo con una fecha fuera de todo el histórico actual:

In [26]:
from src.feature_engineering import construir_features
df_pruebas = pd.DataFrame()
for fecha_prueba in ['2027-02-14', '2027-05-03', '2027-11-26']:
    for tramo_futuro in ['mañana', 'tarde']:
        fila = construir_features(fecha_prueba, tramo_futuro)
        df_pruebas = pd.concat([df_pruebas, fila], ignore_index=True, axis=1)
df_pruebas = df_pruebas.T.sort_values("fecha_cita")  # Transponer para que cada fila sea una fecha/tramo
print(df_pruebas.shape, df_pruebas)

(6, 17)             fecha_cita   tramo dia_semana nombre_dia es_finde mes  anio  \
0  2027-02-14 00:00:00  mañana          6     Sunday     True   2  2027   
1  2027-02-14 00:00:00   tarde          6     Sunday     True   2  2027   
2  2027-05-03 00:00:00  mañana          0     Monday    False   5  2027   
3  2027-05-03 00:00:00   tarde          0     Monday    False   5  2027   
4  2027-11-26 00:00:00  mañana          4     Friday    False  11  2027   
5  2027-11-26 00:00:00   tarde          4     Friday    False  11  2027   

  semana_iso trimestre dias_desde_inicio      grupo_dia  temporada  \
0          6         1              1011  fin_de_semana   invierno   
1          6         1              1011  fin_de_semana   invierno   
2         18         2              1089   entre_semana  primavera   
3         18         2              1089   entre_semana  primavera   
4         47         4              1296        viernes      otoño   
5         47         4              1296      